# BSDSF23M015 - Prompt Engineering Notebook

In [16]:
from openai import OpenAI
import os, csv, re
from dotenv import load_dotenv

# Load .env variables
load_dotenv()

# Configure API access - use environment variables for safety
API_KEY = os.getenv('OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL', 'https://api.groq.com/openai/v1')
if not API_KEY:
    raise ValueError('OPENAI_API_KEY environment variable is required. Set it in your .env file or environment.')

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print('OpenAI client configured: yes')

OpenAI client configured: yes


## Task 1: Parameter Sensitivity Study

We evaluate prompt responses to the same healthcare task under different temperature/top_p/max_tokens/penalty settings and save metrics for each run.

In [17]:
def task1_get_completion(prompt, model='llama-3.1-8b-instant', temperature=0, top_p=1, max_tokens=200, frequency_penalty=0.0, presence_penalty=0.0):
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
    )
    return response.choices[0].message.content

def task1_analyze_output(text):
    clean = text.strip()
    words = re.findall(r'\w+', clean)
    word_count = len(words)
    sentences = [s.strip() for s in re.split(r'[.!?]+', clean) if s.strip()]
    unique_words = set(w.lower() for w in words)
    unique_ratio = len(unique_words) / max(word_count, 1)
    repetition_score = 1.0 - unique_ratio
    healthcare_keywords = [
        'health', 'hospital', 'doctor', 'patient', 'medical', 'diagnosis', 'treatment', 'AI', 'artificial',
        'intelligence', 'therapy', 'disease', 'clinical', 'care',
    ]
    found = sum(1 for k in healthcare_keywords if re.search(rf'\b{re.escape(k)}\b', text, re.IGNORECASE))
    topic_drift = 1.0 - found / len(healthcare_keywords)
    coherence = max(0.0, min(1.0, len(sentences) / max(word_count / 20, 1)))
    creativity = min(1.0, unique_ratio * 1.2)
    return {
        'word_count': word_count,
        'coherence': round(coherence, 3),
        'creativity': round(creativity, 3),
        'repetition': round(repetition_score, 3),
        'topic_drift': round(topic_drift, 3),
    }

def run_task1():
    prompt = 'Write a 150-word explanation of how artificial intelligence is used in healthcare.'
    configs = [
        {'temperature': 0.2, 'top_p': 1.0, 'max_tokens': 180, 'frequency_penalty': 0.0, 'presence_penalty': 0.0},
        {'temperature': 0.8, 'top_p': 0.9, 'max_tokens': 180, 'frequency_penalty': 0.0, 'presence_penalty': 0.0},
        {'temperature': 0.3, 'top_p': 0.5, 'max_tokens': 140, 'frequency_penalty': 0.5, 'presence_penalty': 0.0},
        {'temperature': 0.9, 'top_p': 0.95, 'max_tokens': 210, 'frequency_penalty': 0.0, 'presence_penalty': 0.5},
        {'temperature': 0.6, 'top_p': 0.7, 'max_tokens': 150, 'frequency_penalty': 0.8, 'presence_penalty': 0.3},
        {'temperature': 0.1, 'top_p': 1.0, 'max_tokens': 100, 'frequency_penalty': 0.3, 'presence_penalty': 0.3},
        {'temperature': 0.5, 'top_p': 0.4, 'max_tokens': 170, 'frequency_penalty': 0.2, 'presence_penalty': 0.4},
        {'temperature': 0.99, 'top_p': 0.3, 'max_tokens': 220, 'frequency_penalty': 0.0, 'presence_penalty': 0.8},
    ]
    rows = []
    for i, cfg in enumerate(configs, start=1):
        output = task1_get_completion(prompt, temperature=cfg['temperature'], top_p=cfg['top_p'], max_tokens=cfg['max_tokens'], frequency_penalty=cfg['frequency_penalty'], presence_penalty=cfg['presence_penalty'])
        metrics = task1_analyze_output(output)
        rows.append({
            'experiment': i,
            'temperature': cfg['temperature'],
            'top_p': cfg['top_p'],
            'max_tokens': cfg['max_tokens'],
            'frequency_penalty': cfg['frequency_penalty'],
            'presence_penalty': cfg['presence_penalty'],
            'output': output,
            **metrics,
        })
    fields = ['experiment', 'temperature', 'top_p', 'max_tokens', 'frequency_penalty', 'presence_penalty', 'output', 'word_count', 'coherence', 'creativity', 'repetition', 'topic_drift']
    with open('parameter_sensitivity_results.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return rows

task1_results = run_task1()
print('Task1 completed. Rows:', len(task1_results))

Task1 completed. Rows: 8


## Task 1 Results Preview

In [18]:
import pandas as pd
df1 = pd.read_csv('parameter_sensitivity_results.csv')
df1.head(5)

,experiment,temperature,top_p,max_tokens,frequency_penalty,presence_penalty,output,word_count,coherence,creativity,repetition,topic_drift
0,1,0.2,1.00,180,0.0,0.0,Artificial intelligence (AI) is increasingly b...,149,1.0,0.797,0.336,0.214
1,2,0.8,0.90,180,0.0,0.0,Artificial intelligence (AI) is increasingly b...,152,1.0,0.813,0.322,0.357
2,3,0.3,0.50,140,0.5,0.0,Artificial intelligence (AI) is increasingly b...,117,1.0,0.851,0.291,0.143
3,4,0.9,0.95,210,0.0,0.5,Artificial intelligence (AI) is increasingly b...,162,1.0,0.822,0.315,0.286
4,5,0.6,0.70,150,0.8,0.3,Artificial intelligence (AI) is increasingly b...,124,1.0,0.861,0.282,0.214


## Task 2: Prompt Optimization

We compare several prompt styles (basic, role-based, structured, few-shot, chain-of-thought) and score outputs on clarity, structure, completeness, and factuality.

In [19]:
def task2_get_completion(prompt, model='llama-3.1-8b-instant', temperature=0.3, top_p=0.9, max_tokens=300):
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

def task2_score_output(text):
    if not text:
        return {'clarity': 0.0, 'structure': 0.0, 'completeness': 0.0, 'factuality': 0.0}
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    words = re.findall(r'\w+', text)
    avg_sent_len = len(words) / max(len(sentences), 1)
    clarity = max(0.0, min(1.0, 1.2 - avg_sent_len / 25))
    structure = 0.0
    if '- ' in text or '•' in text or '1.' in text or 'a)' in text:
        structure = 0.9
    elif len(sentences) > 3:
        structure = 0.7
    else:
        structure = 0.4
    key_causes = ['assassination', 'archduke', 'main', 'alliances', 'imperialism', 'militarism', 'nationalism']
    found = sum(1 for w in key_causes if re.search(rf'\b{re.escape(w)}\b', text, re.IGNORECASE))
    completeness = min(1.0, found / 4)
    factual_terms = ['1914', 'Austria', 'Serbia', 'Balkan', 'Triple Entente', 'Central Powers', 'Franz Ferdinand']
    fact_hits = sum(1 for w in factual_terms if re.search(rf'\b{re.escape(w)}\b', text, re.IGNORECASE))
    factuality = min(1.0, fact_hits / 4)
    return {'clarity': round(clarity, 2), 'structure': round(structure, 2), 'completeness': round(completeness, 2), 'factuality': round(factuality, 2)}

def run_task2():
    base_task = 'Explain the causes of World War I.'
    prompts = {
        'basic': base_task,
        'role_based': 'You are a university history professor. Explain the causes of World War I in a concise academic paragraph.',
        'structured': 'You are a university history professor. Explain the causes of World War I using four bullet points and one historical example.',
        'few_shot': 'Example: Q: What caused the American Revolution? A: Tensions over taxation without representation, British trade restrictions, and political autonomy led colonists to revolt. Now, explain the causes of World War I.',
                                    'chain_of_thought': 'Think step by step and explain why World War I began, including political alliances, nationalism, and immediate triggers. Then summarize the main causes.',
        }
    results = []
    for key, prompt in prompts.items():
        output = task2_get_completion(prompt)
        metrics = task2_score_output(output)
        results.append({
            'prompt_type': key,
            'prompt': prompt,
            'output': output,
            **metrics,
        })
    with open('prompt_optimization_results.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['prompt_type', 'prompt', 'output', 'clarity', 'structure', 'completeness', 'factuality'])
        writer.writeheader()
        writer.writerows(results)
    return results

task2_results = run_task2()
print('Task2 completed. Rows:', len(task2_results))

Task2 completed. Rows: 5


### Task 2 Results Preview

In [20]:
df2 = pd.read_csv('prompt_optimization_results.csv')
df2[['prompt_type', 'clarity', 'structure', 'completeness', 'factuality']]

,prompt_type,clarity,structure,completeness,factuality
0,basic,0.56,0.9,1.0,1.0
1,role_based,0.20,0.9,1.0,1.0
2,structured,0.19,0.9,1.0,1.0
3,few_shot,0.67,0.9,1.0,1.0
4,chain_of_thought,0.72,0.9,1.0,1.0


## Task 3: Failure Analysis

In this task we probe hallucination, ambiguity, and confident-but-wrong failures and log analysis guidance.

In [21]:
def task3_get_completion(prompt, model='llama-3.1-8b-instant', temperature=0.4, top_p=0.9, max_tokens=220):
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

def evaluate_failure(output, expected_hallucination=False):
    clean = output.lower()
    ambiguity = 1.0 if any(w in clean for w in ['maybe', 'could', 'possibly', 'not sure']) else 0.0
    incorrect_confidence = 1.0 if expected_hallucination and len(clean) > 20 else 0.0
    return ambiguity, incorrect_confidence

def run_task3():
    prompts = [
        {'name': 'hallucination', 'prompt': 'List five scientific discoveries made by Nikola Tesla in 2005.', 'expected_hallucination': True, 'failure_type': 'hallucination'},
        {'name': 'ambiguous', 'prompt': 'Explain the economic policies of the World Health Organization.', 'expected_hallucination': False, 'failure_type': 'ambiguous'},
        {'name': 'confident_incorrect', 'prompt': 'Name the current president of France in 1890 and explain one policy.', 'expected_hallucination': True, 'failure_type': 'confident_but_incorrect'},
        {'name': 'misleading', 'prompt': 'Describe how the moon landing in 2021 impacted global politics.', 'expected_hallucination': True, 'failure_type': 'hallucination'},
        {'name': 'vague', 'prompt': 'What are the things people do?', 'expected_hallucination': False, 'failure_type': 'ambiguous'},
    ]
    rows = []
    for item in prompts:
        output = task3_get_completion(item['prompt'])
        ambiguity, incorrect_confidence = evaluate_failure(output, item['expected_hallucination'])
        analysis = []
        if item['failure_type'] == 'hallucination':
            analysis.append('Model can invent facts when asked impossible or wrong context.')
            analysis.append('Use verifiable context and correct temporal boundaries.')
        if item['failure_type'] == 'ambiguous':
            analysis.append('Prompt vagueness leads to generic text; refine with constraints and required output format.')
        if item['failure_type'] == 'confident_but_incorrect':
            analysis.append('Model may assert confidently when prompt asks impossible historical facts.')
        rows.append({
            'name': item['name'],
            'prompt': item['prompt'],
            'failure_type': item['failure_type'],
            'output': output,
            'ambiguity': ambiguity,
            'confident_incorrect': incorrect_confidence,
            'analysis': ' '.join(analysis),
        })
    with open('llm_failure_analysis_results.csv', 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['name', 'prompt', 'failure_type', 'output', 'ambiguity', 'confident_incorrect', 'analysis']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return rows

task3_results = run_task3()
print('Task3 completed. Rows:', len(task3_results))

Task3 completed. Rows: 5


### Task 3 Results Preview

In [22]:
df3 = pd.read_csv('llm_failure_analysis_results.csv')
df3[['name', 'failure_type', 'ambiguity', 'confident_incorrect']]

,name,failure_type,ambiguity,confident_incorrect
0,hallucination,hallucination,1.0,1.0
1,ambiguous,ambiguous,0.0,0.0
2,confident_incorrect,confident_but_incorrect,0.0,1.0
3,misleading,hallucination,0.0,1.0
4,vague,ambiguous,0.0,0.0


## Task 4: Multi-step Pipeline

This task runs a simple 4-stage pipeline: summarization, fact extraction, topic classification, and tweet generation.

In [23]:
def task4_get_completion(prompt, model='llama-3.1-8b-instant', temperature=0.3, top_p=0.9, max_tokens=250):
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

def run_task4():
    article = ('A new global health initiative was launched this week to improve access to telemedicine services in rural regions. '
               'The program aims to provide digital consultations, remote diagnostics, and training for local health workers. '
               'Officials say the first phase will focus on maternal and child health and is backed by partnerships across five countries.')
    stage1_prompt = 'Read the following news article and provide a concise 2-sentence summary:' + article
    stage1_output = task4_get_completion(stage1_prompt)
    stage2_prompt = 'Given this summary, extract key facts in a bullet list (who, what, where, why):' + stage1_output
    stage2_output = task4_get_completion(stage2_prompt)
    stage3_prompt = 'Classify the topic into one of [health, technology, politics, environment, business]. Explain in one short sentence.' + stage2_output
    stage3_output = task4_get_completion(stage3_prompt)
    stage4_prompt = 'Using the extracted facts and topic classification, write a tweet-length summary (<= 280 chars): Facts:' + stage2_output + 'Topic:' + stage3_output
    stage4_output = task4_get_completion(stage4_prompt)
    rows = [
        {'stage': '1_summary', 'prompt': stage1_prompt, 'output': stage1_output},
        {'stage': '2_key_facts', 'prompt': stage2_prompt, 'output': stage2_output},
        {'stage': '3_topic_classification', 'prompt': stage3_prompt, 'output': stage3_output},
        {'stage': '4_tweet_summary', 'prompt': stage4_prompt, 'output': stage4_output},
    ]
    with open('multi_step_pipeline_results.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['stage', 'prompt', 'output'])
        writer.writeheader()
        writer.writerows(rows)
    return rows

task4_results = run_task4()
print('Task4 completed. Rows:', len(task4_results))

Task4 completed. Rows: 4


### Task 4 Results Preview

In [24]:
df4 = pd.read_csv('multi_step_pipeline_results.csv')
df4

,stage,prompt,output
0,1_summary,Read the following news article and provide a ...,A new global health initiative has been launch...
1,2_key_facts,"Given this summary, extract key facts in a bul...",Here are the key facts extracted in a bullet l...
2,3_topic_classification,"Classify the topic into one of [health, techno...",The topic can be classified as **health**.
3,4_tweet_summary,Using the extracted facts and topic classifica...,"""Expanding access to healthcare in rural regio..."


## Final Notes

- The notebook runs all four tasks and writes outputs to CSV files.
- If you want to re-run only one task, execute the corresponding function cell (`run_task1`, `run_task2`, `run_task3`, `run_task4`).
- For better security, keep your API key out of source control by using environment variables.